# Listed SPX RR-spread: development feasibility

Mechanics and coverage only, using **2023–2024 option chains**. No strategy P&L is computed. No 2025 option file is opened. The [frozen trading protocol](../docs/trading_protocol.md) governs selection and lifecycle rules; forecasting results are unchanged.

The eventual signal is the existing static M2 for $RR25_{30}-RR25_{60}$. Signal at EOD $t$ enters at $t+1$ and exits at $t+5$. Here every development entry session is checked with a canonical positive direction, independently of signal availability, to measure mechanical feasibility without simulating trades.

Aggregate outputs are retained; private contract examples write to a gitignored CSV.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.forecasting import DEV_START, DEV_END
from src.time_series import build_daily_time_series
from src.trade_selection import fit_rr_spread_signal
from src.trading_feasibility import run_development_feasibility

raw_files = [ROOT / "data/raw/spx_option_prices_2023.csv",
             ROOT / "data/raw/spx_option_prices_2024.csv"]
spx = pd.read_csv(ROOT / "data/raw/spx_security_prices.csv", parse_dates=["date"])
spx = spx.loc[spx.date <= DEV_END].copy()
metrics = pd.read_csv(
    ROOT / "data/processed/spx_skew_metrics_2023_2025.csv", parse_dates=["quote_date"]
)
metrics = metrics.loc[metrics.quote_date.between(DEV_START, DEV_END)].copy()
assert metrics.quote_date.max() <= DEV_END and spx.date.max() <= DEV_END
daily_surface = build_daily_time_series(metrics, spx)
frozen_signal = fit_rr_spread_signal(daily_surface, spx)
display(pd.DataFrame([frozen_signal]))

,intercept,slope,n_train,fit_end,last_matured_origin
0,0.000654,-0.001725,426,2024-12-31,2024-12-23


## Stream the development chains

Reuse parity-OLS forwards, midpoint IV solvers and Black Greeks. Expiry errors are limited to seven calendar days, signed forward delta errors to 0.05. Entry quotes need positive bid and finite noncrossed ask. No volume/OI/vendor-IV cutoffs or new spread screen.

Only selected contracts and their availability flags are retained across dates. The runner never calls the accounting/P&L API. A missing chain stays a missing session rather than compressing the calendar.

In [2]:
feasibility = run_development_feasibility(raw_files, spx, progress=True)
sessions = feasibility["daily"]
legs = feasibility["legs"]
wings = feasibility["wings"]
lifecycle = feasibility["lifecycle"]
assert sessions.quote_date.max() <= DEV_END
assert legs.quote_date.max() <= DEV_END
display(pd.DataFrame({
    "sessions": [len(sessions)],
    "first_date": [sessions.quote_date.min()],
    "last_date": [sessions.quote_date.max()],
    "raw_rows_checked": [sessions.raw_rows.sum()],
    "PM_rows_checked": [sessions.pm_rows.sum()],
}))

2023-02-07: 25/502 development sessions checked


2023-03-15: 50/502 development sessions checked


2023-04-20: 75/502 development sessions checked


2023-05-25: 100/502 development sessions checked


2023-07-03: 125/502 development sessions checked


...

2024-09-11: 425/502 development sessions checked


2024-10-16: 450/502 development sessions checked


2024-11-20: 475/502 development sessions checked


2024-12-27: 500/502 development sessions checked


,sessions,first_date,last_date,raw_rows_checked,PM_rows_checked
0,502,2023-01-03,2024-12-31,10659814,7268276


## Expiry coverage and four-wing availability

Coverage denominators are all development SPX sessions. Expiry pairing is determined first; a failed delta wing does not trigger a search for a farther expiry. Distances below describe all selected pairs / all nearest-wing candidates, including failures. Poor coverage is a release-gate issue, not a reason to loosen the frozen rules.

In [3]:
coverage = pd.DataFrame([
    {"stage": "eligible expiry pair", "n": int(sessions.expiry_pair_available.sum())},
    {"stage": "all four 25-delta wings", "n": int(sessions.four_wings_available.sum())},
])
coverage["percent_of_sessions"] = 100 * coverage.n / len(sessions)
display(coverage)
display(sessions[["dte_error_30", "dte_error_60"]].describe(percentiles=[.05, .5, .95]))
wing_summary = wings.groupby(["target_days", "option_type"]).agg(
    candidates=("eligible", "size"), accepted=("eligible", "sum"),
    median_delta_distance=("delta_distance", "median"),
    max_delta_distance=("delta_distance", "max"),
)
wing_summary["rejected_by_005_rule"] = wing_summary.candidates - wing_summary.accepted
display(wing_summary)
display(wings.delta_distance.describe(percentiles=[.05, .5, .95]))
display(sessions.reason.value_counts().rename("sessions"))

,stage,n,percent_of_sessions
0,eligible expiry pair,492,98.007968
1,all four 25-delta wings,481,95.816733


,dte_error_30,dte_error_60
count,492.000000,492.000000
mean,0.455285,3.060976
std,0.549053,1.929602
min,0.000000,0.000000
5%,0.000000,0.000000
50%,0.000000,3.000000
95%,1.000000,7.000000
max,2.000000,7.000000


candidates  accepted  median_delta_distance  \
target_days option_type                                                
30          call                492       482               0.004580   
            put                 492       492               0.002108   
60          call                492       491               0.002447   
            put                 492       492               0.001161   

                         max_delta_distance  rejected_by_005_rule  
target_days option_type                                            
30          call                   0.107888                    10  
            put                    0.015312                     0  
60          call                   0.054259                     1  
            put                    0.011355                     0

count    1968.000000
mean        0.004674
std         0.008557
min         0.000006
5%          0.000188
50%         0.002077
95%         0.019276
max         0.107888
Name: delta_distance, dtype: float64

reason
                         481
missing_25_delta_wing     11
no_expiry_pair            10
Name: sessions, dtype: int64

## Quote quality

Widths are in SPX premium points, not volatility. Relative width is $(ask-bid)/midpoint$. Zero-bid and invalid-quote rejection counts refer to the whole PM chain, not only the four selected wings. Zero bids can contribute to parity inference but cannot be entry contracts; vendor Greeks/IV are never used for selection.

In [4]:
legs["quote_width"] = legs.best_ask - legs.best_bid
legs["relative_width"] = legs.quote_width / legs.mid_price
display(legs[["quote_width", "relative_width", "best_bid", "best_ask"]].describe(
    percentiles=[.05, .5, .95]
))
display(sessions[["zero_bid_rows", "invalid_quote_rows", "iv_failures",
                  "failed_forward_expiries"]].agg(["sum", "max"]))

,quote_width,relative_width,best_bid,best_ask
count,1924.000000,1924.000000,1924.000000,1924.000000
mean,0.523077,0.015439,35.064917,35.587994
std,0.375065,0.009342,11.646325,11.763190
min,0.100000,0.004090,14.600000,15.000000
5%,0.300000,0.007398,19.815000,20.200000
50%,0.400000,0.013947,33.100000,33.500000
95%,0.900000,0.026667,57.085000,57.985000
max,6.900000,0.158273,106.400000,108.800000


,zero_bid_rows,invalid_quote_rows,iv_failures,failed_forward_expiries
sum,377894,2,0,0
max,1129,1,0,0


## Position properties (not returns)

For the canonical positive signal: long 30D put / short 30D call / short 60D put / long 60D call. Each wing contributes 0.25 absolute entry vega; gross vega is one and net entry vega is zero. Contract multiplier 100 is used consistently in cash, vega and spot delta. Quantities are **fractional theoretical contracts**; this is not a capital-normalised or integer-order execution claim.

The SPX hedge uses spot sensitivity $DF/S$ times signed forward delta. It is a frictionless index-level proxy, not a costed futures hedge.

In [5]:
position = sessions.loc[sessions.four_wings_available,
                        ["gross_vega", "net_vega", "portfolio_delta", "initial_hedge"]].copy()
position["absolute_hedge_size"] = position.initial_hedge.abs()
display(position.describe(percentiles=[.05, .5, .95]))
display(legs.groupby(["target_days", "option_type"]).quantity.describe())
assert np.allclose(position.gross_vega, 1)
assert np.allclose(position.net_vega, 0)
assert np.allclose(legs.gross_vega, .25)

,gross_vega,net_vega,portfolio_delta,initial_hedge,absolute_hedge_size
count,4.810000e+02,4.810000e+02,481.000000,481.000000,481.000000
mean,1.000000e+00,2.538972e-18,-0.000084,0.000084,0.000084
std,5.067452e-18,1.582185e-17,0.000014,0.000014,0.000014
min,1.000000e+00,-2.775558e-17,-0.000120,0.000053,0.000053
5%,1.000000e+00,-2.775558e-17,-0.000108,0.000063,0.000063
50%,1.000000e+00,0.000000e+00,-0.000082,0.000082,0.000082
95%,1.000000e+00,2.775558e-17,-0.000063,0.000108,0.000108
max,1.000000e+00,5.551115e-17,-0.000053,0.000120,0.000120


count      mean           std       min       25%  \
target_days option_type                                                      
30          call         481.0 -0.000006  7.948092e-07 -0.000008 -0.000006   
            put          481.0  0.000006  7.696471e-07  0.000004  0.000005   
60          call         481.0  0.000004  5.580182e-07  0.000003  0.000004   
            put          481.0 -0.000004  5.551694e-07 -0.000005 -0.000005   

                              50%       75%       max  
target_days option_type                                
30          call        -0.000006 -0.000005 -0.000004  
            put          0.000006  0.000006  0.000007  
60          call         0.000004  0.000005  0.000005  
            put         -0.000004 -0.000004 -0.000003

## Lifecycle coverage only

Track the **same option IDs** through the scheduled exit. The whole basket closes at $t+5$, or at $t+6$ once if all four executable quotes are then available. No price changes or hedge P&L are calculated here. Boundary entries whose origin/exit/fallback is outside development are excluded from lifecycle denominators and reported separately.

Complete quote-path coverage is stricter than exit coverage. Daily hedge inputs must also exist: a held zero-bid option can supply a valid midpoint Greek, but a missing quote/IV cannot silently carry a stale hedge.

In [6]:
exit_counts = lifecycle.status.value_counts().reindex(
    ["scheduled", "fallback", "unevaluable"], fill_value=0
).rename("baskets").to_frame()
exit_counts["percent"] = 100 * exit_counts.baskets / len(lifecycle)
display(exit_counts)
display(pd.DataFrame([{
    "lifecycle_eligible_baskets": len(lifecycle),
    "boundary_entries": int(sessions.lifecycle_boundary.sum()),
    "scheduled_contract_quote_percent": 100 * lifecycle.scheduled_contracts.sum() / (4 * len(lifecycle)),
    "contract_quote_path_percent": 100 * lifecycle.contracts_valid_through_scheduled.sum() / (4 * len(lifecycle)),
    "complete_basket_quote_path_percent": 100 * lifecycle.complete_quote_path.mean(),
    "complete_hedge_input_percent": 100 * lifecycle.complete_hedge_inputs.mean(),
}]))
display(lifecycle.groupby("offset").status.value_counts().unstack(fill_value=0))

,baskets,percent
status,,
scheduled,475,100.0
fallback,0,0.0
unevaluable,0,0.0


,lifecycle_eligible_baskets,boundary_entries,scheduled_contract_quote_percent,contract_quote_path_percent,complete_basket_quote_path_percent,complete_hedge_input_percent
0,475,6,100.0,100.0,100.0,100.0


status,scheduled
offset,
0,97
1,95
2,91
3,94
4,98


## Mechanically chosen examples

First entry session with all four wings in each development calendar quarter. Dates are not chosen for attractive quotes, smile shape or realised outcomes. These are contract-selection details, not trade results.

In [ ]:
valid_dates = sessions.loc[sessions.four_wings_available, "quote_date"]
example_dates = valid_dates.groupby(valid_dates.dt.to_period("Q")).min()
example_columns = [
    "quote_date", "target_days", "option_id", "expiry_date", "days_to_expiry",
    "strike", "option_type", "best_bid", "best_ask", "mid_price", "mid_iv",
    "forward_delta", "delta_distance", "vega", "quantity", "gross_vega",
    "net_vega", "option_delta", "position_delta",
]
example_path = ROOT / "data/processed/trading_feasibility_examples.csv"
legs.loc[legs.quote_date.isin(example_dates), example_columns].to_csv(example_path, index=False)
print("Private contract examples saved locally; omitted from public outputs.")

## Release gate

The coverage tables above are the only real-option-data outputs of this notebook. Any unusable expiry/wing availability or widespread missing lifecycle inputs must be reviewed **without loosening rules**. No aggregate strategy P&L, price-change attribution, hit rate, Sharpe, 2025 selection or profitability conclusion is produced. A locked 2025 trading run requires separate explicit approval.

Development results: expiry pairs on 492/502 sessions (98.01%); four-wing baskets on 481/502 (95.82%). Ten dates lack a pair and eleven fail the delta rule, predominantly 30D calls. Median DTE errors are 0 and 3 days for the near/far tenors; the far-tenor 95th percentile reaches the frozen seven-day limit. All 475 lifecycle-eligible baskets have full quote-path and hedge-input coverage, exit on schedule, and need no fallback. Six selected dates are boundary-only observations. These findings support mechanical feasibility without relaxing any rule; they say nothing about profitability. The protocol is frozen and ready for a separately authorised locked trading run.